In [ ]:
import pandas as pd

# Load the CSV file
csv_file = '/Users/marcelosilva/Desktop/early-obesity-prediction/B-Featuring Removing/1missing.csv'
df_csv = pd.read_csv(csv_file)

# Load the Excel file (dictionary)
excel_file = '/Users/marcelosilva/Desktop/early-obesity-prediction/4-Dicionario-ENANI-2019 (1).xlsx'
df_dict = pd.read_excel(excel_file)

# Get all columns from the CSV
csv_columns = df_csv.columns.tolist()

# Create list to store matches
matches = []

# Cross-reference column names with dictionary variables
for column in csv_columns:
    # Search for matches (case insensitive)
    match = df_dict[df_dict['variavel'].str.lower() == column.lower()]
    
    if not match.empty:
        # If match found, add to list
        description = match['descricao'].iloc[0] if 'descricao' in df_dict.columns else 'Description not found'
        matches.append(f"{column}: {description}")

# Save results to 2dict.txt file
output_file = '2dict.txt'
with open(output_file, 'w', encoding='utf-8') as f:
    for match in matches:
        f.write(match + '\n')

print(f"File {output_file} created with {len(matches)} correlations found.")
print(f"Total columns in CSV: {len(csv_columns)}")

In [ ]:
import pandas as pd
import os

# List of selected variables for analysis of the first 24 months of life
selected_variables = [
    # Identification and demographic characteristics
    'id_anon',                    # Child identification code
    'a00_regiao',                 # Macro-region
    'b02_sexo',                   # Child's sex
    'b04_idade',                  # Child's age in complete years
    'bb04_idade_da_mae',          # Mother's age in complete years
    'd01_cor',                    # Child's color or race
    
    # Perinatal and birth data
    'h01_semanas_gravidez',       # How many weeks of pregnancy when the child was born
    'h02_peso',                   # Birth weight (in grams) of the child
    'h03_altura',                 # How many centimeters was the child at birth
    'h04_parto',                  # Child's delivery type
    'h05_chupeta_usou',           # From the child's birth until today, has he/she used a pacifier
    
    # Congenital conditions and syndromes
    'h10b_sindrome_de_down',      # Has a doctor ever said the child has Down syndrome, cystic fibrosis, phenylketonuria, or autism
    'h10b1_sindrome_nao',         # Does not have a syndrome
    'h10b2_sindrome_down',        # Down syndrome
    'h10b3_sindrome_fibrose',     # Cystic fibrosis
    'h10b4_sindrome_fenilcetonuria', # Phenylketonuria
    'h10b5_sindrome_autismo',     # Autism
    'h10b9_sindrome_nao_sabe',    # Does not know
    
    # Maternal characteristics - education
    'j08_ler',                    # Can the mother/guardian read and write
    'j09_frequenta',              # Does/did the mother/guardian attend school
    'j10_serie',                  # Last grade/year the mother/guardian completed
    
    # Prenatal and pregnancy
    'k03_prenatal',               # Had prenatal care for the child
    'k04_prenatal_semanas',       # How many weeks of pregnancy when prenatal care started for the child
    'k05_prenatal_consultas',     # How many prenatal consultations during pregnancy of the child
    'k06_peso_engravidar',        # Weight before becoming pregnant with the child
    'k07_peso_final',             # Weight before delivery (at end of pregnancy) of the child
    'k08_quilos',                 # How many kilograms were gained during pregnancy of the child
    
    # Breastfeeding and early feeding practices
    'k11_amamentou',              # Has the child ever been breastfed
    'k12_tempo',                  # How long after birth was the child put to the breast for the first time
    'k13_tempo_medida',           # Hours or days
    'k15_recebeu',                # During the stay at the maternity ward, did the child receive any milk other than breast milk
    'k16_liquido',                # In the first days after delivery, before the milk came in, was any other liquid given to the child besides breast milk
    'k18_somente',                # For how long was only breast milk given to the child, without water, tea, juice, or other foods
    'k19_somente_medida',         # Days or months
    'k20_doou',                   # Since breastfeeding the child, have you donated milk to a human milk bank or collection point
    'k21_recebeu',                # Since breastfeeding the child, have you ever received milk from a human milk bank
    'k22_amamentou',              # Since breastfeeding the child, have you breastfed another woman's child
    'k23_deixou',                 # Since breastfeeding the child, have you let your child be breastfed by another woman
    'k24_utilizou',               # Since breastfeeding the child, have you used
    'k241_utilizou_concha',       # Used breast shell
    'k242_utilizou_protetor',     # Used nipple shield
    'k243_utilizou_bico',         # Used artificial nipple
    'k244_utilizou_bomba',        # Used breast pump
    'k245_utilizou_mamadeira',    # Used baby bottle
    'k246_utilizou_sondinha',     # Used feeding tube
    'k247_utilizou_copo',         # Used cup
    'k248_utilizou_nao',          # Did not use any
    'k249_utilizou_nao_sabe',     # Does not know
    'k25_mamadeira',              # Does the child use or has the child ever used a baby bottle
    'k28_aleitamento',            # Do you access or have you accessed the internet to search for information about breastfeeding
    
    # Socioeconomic and equity variables
    'q01_recebe_beneficio',       # Does anyone in the household receive government benefit(s)
    
    # Current maternal anthropometry (for feature engineering)
    't02_peso_medida1',           # Mother's weight measurement 1 (kg)
    't03_peso_medida2',           # Mother's weight measurement 2 (kg)
    
    # Socioeconomic indicators
    'vd_ien_escore',              # National Economic Indicator (IEN) score
    
    # Target/outcome variables
    'vd_zimc',                    # BMI-for-age z-score of the child
    'vd_prematura_igb'            # Premature children with gestational age between 189 and 454 days
]

def filter_dataset():
    """
    Filters the original dataset keeping only the selected variables
    and generates a report of the changes
    """
    
    # Define paths
    input_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/B-Featuring Removing/1missing.csv'
    output_csv_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/B-Featuring Removing/2features24.csv'
    output_txt_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/B-Featuring Removing/2features24.txt'
    
    try:
        # Load the original dataset
        print("Loading original dataset...")
        df_original = pd.read_csv(input_path)
        
        # Check which selected variables exist in the dataset
        available_vars = [var for var in selected_variables if var in df_original.columns]
        missing_vars = [var for var in selected_variables if var not in df_original.columns]
        
        # Filter dataset with available variables
        df_filtered = df_original[available_vars].copy()
        
        # Save new CSV
        print("Saving filtered dataset...")
        df_filtered.to_csv(output_csv_path, index=False)
        
        # Calculate statistics
        total_original = len(df_original.columns)
        total_selecionadas = len(available_vars)
        total_excluidas = total_original - total_selecionadas
        
        # Identify excluded variables
        excluded_vars = [col for col in df_original.columns if col not in available_vars]
        
        # Create report in TXT file
        print("Generating report...")
        with open(output_txt_path, 'w', encoding='utf-8') as f:
            f.write("VARIABLE SELECTION REPORT - FIRST 24 MONTHS OF LIFE\n")
            f.write("=" * 70 + "\n\n")
            
            f.write("STATISTICAL SUMMARY:\n")
            f.write("-" * 20 + "\n")
            f.write(f"Total columns in original dataset (1missing.csv): {total_original}\n")
            f.write(f"Total selected variables: {len(selected_variables)}\n")
            f.write(f"Total available and kept variables: {total_selecionadas}\n")
            f.write(f"Total excluded variables: {total_excluidas}\n")
            f.write(f"Original dataset dimensions: {df_original.shape}\n")
            f.write(f"Filtered dataset dimensions: {df_filtered.shape}\n\n")
            
            if missing_vars:
                f.write("SELECTED VARIABLES NOT FOUND IN DATASET:\n")
                f.write("-" * 50 + "\n")
                for i, var in enumerate(missing_vars, 1):
                    f.write(f"{i:2d}. {var}\n")
                f.write("\n")
            
            f.write("VARIABLES KEPT IN FILTERED DATASET:\n")
            f.write("-" * 40 + "\n")
            for i, var in enumerate(available_vars, 1):
                f.write(f"{i:2d}. {var}\n")
            
            f.write(f"\n\nVARIABLES EXCLUDED FROM ORIGINAL DATASET ({total_excluidas}):\n")
            f.write("-" * 50 + "\n")
            for i, var in enumerate(excluded_vars, 1):
                f.write(f"{i:3d}. {var}\n")
        
        # Final summary
        print("\n" + "="*50)
        print("PROCESSING COMPLETED SUCCESSFULLY!")
        print("="*50)
        print(f"Original dataset: {df_original.shape[0]} rows x {df_original.shape[1]} columns")
        print(f"Filtered dataset: {df_filtered.shape[0]} rows x {df_filtered.shape[1]} columns")
        print(f"Variables removed: {total_excluidas}")
        print(f"\nGenerated files:")
        print(f"  Filtered CSV: 2features24.csv")
        print(f"  Report: 2features24.txt")
        
        if missing_vars:
            print(f"\nWARNING: {len(missing_vars)} selected variable(s) were not found in the original dataset.")
            print("Check the report for details.")
        
        return df_filtered
        
    except FileNotFoundError:
        print(f"Error: File not found at {input_path}")
        print("Check if the path is correct.")
        return None
        
    except Exception as e:
        print(f"Error during processing: {str(e)}")
        return None

# Execute processing
if __name__ == "__main__":
    df_resultado = filter_dataset()

## 🔍 Variable Selection for First 24 Months of Life Analysis

This cell implements a function to **filter the original dataset** keeping only the most relevant variables for analyzing child development in the first 24 months of life.

### 📋 Selected Variables

The set of 56 variables was carefully chosen covering:

- **Identification and Demographics**: Child ID, region, sex, age, maternal characteristics
- **Perinatal Data**: gestational weeks, birth weight/height, delivery type
- **Congenital Conditions**: syndromes and special conditions
- **Maternal Characteristics**: education, literacy
- **Prenatal and Pregnancy**: consultations, maternal weight, weight gain
- **Breastfeeding**: breastfeeding and early feeding practices
- **Socioeconomic Indicators**: benefits and economic status
- **Anthropometry and Outcomes**: maternal BMI, child BMI z-scores, prematurity indicators

### 🎯 Main Features

- **Comprehensive Coverage**: Variables spanning from pregnancy to 24 months
- **Evidence-Based Selection**: Focus on factors known to influence early childhood development
- **Quality Control**: Automated filtering with detailed reporting
- **Output Generation**: Creates both filtered dataset (2features24.csv) and detailed report (2features24.txt)